In [0]:
from pyspark.sql.functions import lit

catalog_name = "automating_scd"

bronze_schema = "automating_scd_1_bronze"
silver_schema = "automating_scd_2_silver"
gold_schema = "automating_scd_3_gold"

volume_name = "customer_source_files"

source_path = "/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily"
sink_path = f"/Volumes/{catalog_name}/{bronze_schema}"

date_on_file = "2025-11-01"
file_name = f"customer_changes_{date_on_file}.json"

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{bronze_schema}.{volume_name}")

In [0]:

dbutils.fs.cp(f"{source_path}/{file_name}", f"{sink_path}/{volume_name}/{file_name}")


In [0]:
## Check file
sql_result = (
    spark.sql(f"LIST '{sink_path}/{volume_name}/'")
    .withColumn('volume', lit(f"{volume_name}"))
)

display(sql_result)

In [0]:
sql_query_result = (
    spark.sql(f"""
        SELECT '{volume_name}' as volume_name,
            COUNT(*) as total_rows,
            _metadata.file_name as file_name
        FROM read_files('{sink_path}/{volume_name}')
        GROUP BY _metadata.file_name
    """)
)

display(sql_query_result)

In [0]:
spark.sql(f"""
    DESCRIBE TABLE EXTENDED {catalog_name}.{bronze_schema}.customers_scd_type2_bronze_clean
""").display()

In [0]:
spark.sql(f"""
    DESCRIBE TABLE EXTENDED {catalog_name}.{silver_schema}.customers_scd_type2_silver
""").display()

In [0]:
%sql
SELECT * FROM automating_scd.automating_scd_1_bronze.customers_scd_type2_bronze
where customer_id IN ('CUST_02795','CUST_00453','CUST_01630')

In [0]:
%sql
SELECT * FROM automating_scd.automating_scd_1_bronze.customers_scd_type2_bronze_clean
where customer_id IN 
--('CUST_00326','CUST_00453','CUST_00481','CUST_00496','CUST_00618','CUST_00858','CUST_00970','CUST_01123','CUST_01124','CUST_01354')
('CUST_02795','CUST_00453','CUST_01630','CUST_00858','CUST_00970') 

In [0]:
%sql
SELECT * FROM automating_scd.automating_scd_2_silver.customers_scd_type2_silver
where customer_id IN 
--('CUST_00326','CUST_00453','CUST_00481','CUST_00496','CUST_00618','CUST_00858','CUST_00970','CUST_01123','CUST_01124','CUST_01354')
('CUST_02795','CUST_00453','CUST_01630','CUST_00858','CUST_00970')

In [0]:
%sql
SELECT * FROM automating_scd.automating_scd_3_gold.current_customers_gold
where customer_id IN 
--('CUST_00326','CUST_00453','CUST_00481','CUST_00496','CUST_00618','CUST_00858','CUST_00970','CUST_01123','CUST_01124','CUST_01354')
('CUST_02795','CUST_00453','CUST_01630','CUST_00858','CUST_00970')

In [0]:
%sql
SELECT concat("'",customer_id,"',") AS new_customer_id
FROM automating_scd.automating_scd_2_silver.customers_scd_type2_silver
WHERE `__END_AT` IS NOT NULL;